Amardeep Singh

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import re

# ==============================
# 1. DATA PREPROCESSING
# ==============================

pairs = [
    ("hello", "hi"),
    ("how are you", "i am fine"),
    ("what is your name", "i am chatbot"),
    ("bye", "goodbye")
]

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text

pairs = [(clean_text(i), clean_text(o)) for i, o in pairs]

# Vocabulary
words = []
for p in pairs:
    words += p[0].split()
    words += p[1].split()

vocab = ["<pad>", "<sos>", "<eos>", "<unk>"] + list(set(words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

def encode(sentence):
    return [word2idx.get(w, word2idx["<unk>"]) for w in sentence.split()]

# ==============================
# 2. MODEL
# ==============================

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        embed = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embed)
        return outputs, hidden, cell


class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [1, batch, hidden]
        hidden = hidden[-1]  # [batch, hidden]

        batch_size = encoder_outputs.shape[0]
        seq_len = encoder_outputs.shape[1]

        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=1)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(hidden_size + embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)
        self.attention = Attention(hidden_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(1)  # [batch, 1]
        embed = self.embedding(x)

        attn_weights = self.attention(hidden, encoder_outputs)
        attn_weights = attn_weights.unsqueeze(1)

        context = torch.bmm(attn_weights, encoder_outputs)

        lstm_input = torch.cat((embed, context), dim=2)

        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        output = torch.cat((output.squeeze(1), context.squeeze(1)), dim=1)

        prediction = self.fc(output)

        return prediction, hidden, cell


# ==============================
# 3. INITIALIZE
# ==============================

INPUT_DIM = len(vocab)
EMBED_SIZE = 64
HIDDEN_SIZE = 128

encoder = Encoder(INPUT_DIM, EMBED_SIZE, HIDDEN_SIZE)
decoder = Decoder(INPUT_DIM, EMBED_SIZE, HIDDEN_SIZE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)

# ==============================
# 4. TRAINING
# ==============================

EPOCHS = 500

for epoch in range(EPOCHS):
    total_loss = 0

    for inp, out in pairs:
        inp_tensor = torch.tensor([encode(inp)])  # [1, seq_len]
        out_tensor = torch.tensor(encode(out) + [word2idx["<eos>"]])

        encoder_outputs, hidden, cell = encoder(inp_tensor)

        decoder_input = torch.tensor([word2idx["<sos>"]])

        loss = 0

        for t in range(len(out_tensor)):
            output, hidden, cell = decoder(decoder_input, hidden, cell, encoder_outputs)

            target = out_tensor[t].view(1)   # ✅ FIX
            loss += criterion(output, target)

            decoder_input = target           # ✅ FIX

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# ==============================
# 5. CHAT FUNCTION
# ==============================

def generate_response(sentence):
    sentence = clean_text(sentence)
    inp_tensor = torch.tensor([encode(sentence)])

    encoder_outputs, hidden, cell = encoder(inp_tensor)

    decoder_input = torch.tensor([word2idx["<sos>"]])

    result = []

    for _ in range(10):
        output, hidden, cell = decoder(decoder_input, hidden, cell, encoder_outputs)
        top1 = output.argmax(1)

        word = idx2word[top1.item()]

        if word == "<eos>":
            break

        result.append(word)
        decoder_input = top1

    return " ".join(result)

# ==============================
# 6. TEST
# ==============================

while True:
    user_input = input("You: ")
    if user_input == "exit":
        break
    print("Bot:", generate_response(user_input))

Epoch 0, Loss: 35.2564
Epoch 50, Loss: 0.0854
Epoch 100, Loss: 0.0269
Epoch 150, Loss: 0.0134
Epoch 200, Loss: 0.0081
Epoch 250, Loss: 0.0055
Epoch 300, Loss: 0.0040
Epoch 350, Loss: 0.0030
Epoch 400, Loss: 0.0023
Epoch 450, Loss: 0.0019
You: how are you
Bot: i am fine
You: hello
Bot: hi
You: ok
Bot: goodbye
You: goodbye
Bot: goodbye
You: exit
